# Python or Rust: which implementation to use

This tool ships twice, in Python and in Rust. Both answer to the same
conformance corpus, so on every input that corpus covers they emit identical
bytes. This notebook re-checks that on each run rather than taking it on trust,
because it is the assumption under everything else: only if the two agree byte
for byte is choosing between them a question about cost alone.

**Every number and every recommendation below is computed by the cell above
it.** Nothing quantitative is written into the prose here, and neither is any
claim about the state of the repository. A figure typed into a sentence stops
matching the moment somebody re-runs the notebook, and a recommendation typed
into a sentence stops matching the moment the thing it describes is changed.
The prose says what is being measured and why; the outputs say what came out.

Run the notebook and read the last cell for the recommendation.

In [1]:
import hashlib
import platform
import re
import statistics
import subprocess
import sys
import time
from dataclasses import dataclass
from pathlib import Path


def git(*argv):
    """Return stripped stdout from a git command, or an empty string on failure."""
    return subprocess.run(
        ['git', *argv], capture_output=True, text=True, check=False
    ).stdout.strip()


REPO = Path(git('rev-parse', '--show-toplevel'))
PYTHON_CLI = REPO / '.venv/bin/unwrap-markdown-prose-py'
RUST_CLI = REPO / 'target/release/unwrap-markdown-prose-rs'

for path in (PYTHON_CLI, RUST_CLI):
    if not path.exists():
        raise SystemExit(f'missing {path}: run `uv sync` and `cargo build --release`')

# Read rather than remembered. The release profile is tuned for binary size, so
# every Rust figure below is a size-optimized figure, and a reader comparing
# against their own build needs to know which knob produced these numbers.
level = re.search(
    r'^\s*opt-level\s*=\s*(\S+)', (REPO / 'Cargo.toml').read_text(), re.MULTILINE
)
OPT_LEVEL = level.group(1).strip('"') if level else 'default'

# `release.yml` publishes the prebuilt binaries and a tag is what triggers it.
# Several rows of the recommendation are gated on a release existing, so the
# state is captured here rather than assumed. Local tags are the proxy: this
# notebook does not reach the network, and a tag is the trigger either way.
TAGS = git('tag', '--list', 'v*').split()

rustc = subprocess.run(
    ['rustc', '--version'], capture_output=True, text=True, check=False
).stdout.strip()

print(f'machine   {platform.machine()}  {platform.system()} {platform.release()}')
print(f'python    {platform.python_version()}')
print(f'rust      {rustc}')
print(f'revision  {git("rev-parse", "--short", "HEAD")}')
print(
    f'binary    {RUST_CLI.stat().st_size / 1024:.0f} KB at opt-level {OPT_LEVEL!r}'
    + (' -- tuned for size, not speed' if OPT_LEVEL in {'z', 's'} else '')
)
print(f'releases  {", ".join(TAGS) if TAGS else "none yet; this gates the last cell"}')

machine   arm64  Darwin 23.5.0
python    3.10.18
rust      rustc 1.86.0 (05f9846f8 2025-03-31)
revision  7df08f8
binary    344 KB at opt-level 'z' -- tuned for size, not speed
releases  none yet; this gates the last cell


## Method

Each measurement times a whole process invocation, because a process invocation
is what a hook or a CI step actually pays for. Three warm-up runs are discarded
and the rest are kept.

**Minimum and median are both reported.** The minimum is the honest figure for
a single run: it is the one least disturbed by scheduling, and process startup
has no legitimate source of variance that would make a slower run more
representative. The median sits beside it so that a gap between the two is
visible rather than hidden, and a cell below turns that gap into a check rather
than something a reader has to notice.

**The cost of starting a process at all is measured, not subtracted.** Every
figure here includes fork, exec, and this harness's own pipe setup on top. That
floor is a rounding error against the Python arm and a large fraction of the
Rust one, which is exactly the situation in which a ratio misleads. Subtracting
it is worse than reporting it: the correction is a difference of two similar
numbers, so it carries an uncertainty comparable to the result, and the next
cell shows the candidate reference processes disagreeing with each other by
more than the whole Rust runtime. So the floor is measured, plotted alongside,
and used to mark any ratio whose denominator is sitting on it as a lower bound.

**The two implementations are checked for agreement inside the benchmark.** The
timing loop keeps stdout and exit codes and compares them across the pair. A
binary that rejected its arguments and exited immediately would otherwise post
a spectacular result, and the premise this whole document rests on would go
unexamined in the one place it is free to examine.

In [2]:
WARMUP, REPS = 3, 30

# Percent by which the median may sit above the minimum before a measurement is
# called noisy. Chosen from the startup rows, which are the most repeatable
# thing here and land a few percent apart on an idle machine.
SPREAD_LIMIT = 15.0

# How close to the spawn floor the faster arm may sit before its ratio is
# reported as a bound rather than a value.
FLOOR_FACTOR = 2.0

# Marks a ratio as a lower bound. Named rather than written inline: an escape
# cannot appear inside an f-string expression before Python 3.12.
BOUND = '\u2265'


@dataclass(frozen=True, slots=True)
class Timing:
    """Every sample from one measured command, and what those runs returned."""

    samples: list[float]
    codes: frozenset[int]
    # One digest per repetition rather than one kept stdout. Two runs of the
    # same command returning different bytes means something moved underneath
    # the benchmark, and a single retained stdout cannot show that -- it was
    # the last run, and the last run is the one most likely to look fine.
    digests: frozenset[str]

    @property
    def min(self):
        """The run least disturbed by scheduling."""
        return min(self.samples)

    @property
    def median(self):
        return statistics.median(self.samples)

    @property
    def spread(self):
        """How far the median sits above the minimum, as a percentage."""
        return (self.median - self.min) / self.min * 100


def measure(argv, reps=REPS):
    """Time `reps` whole invocations of `argv` after `WARMUP` discarded runs."""
    for _ in range(WARMUP):
        subprocess.run(argv, capture_output=True, check=False)
    samples, codes, digests = [], set(), set()
    for _ in range(reps):
        started = time.perf_counter()
        done = subprocess.run(argv, capture_output=True, check=False)
        samples.append((time.perf_counter() - started) * 1000)
        codes.add(done.returncode)
        digests.add(hashlib.sha256(done.stdout).hexdigest())
    return Timing(samples, frozenset(codes), frozenset(digests))


# Three trivial programs, timed through the same harness as everything else.
# The cheapest is taken as the floor. They are all reported because their
# disagreement is the argument against subtracting any one of them.
REFERENCES = [['/bin/echo', 'x'], ['/usr/bin/true'], ['/usr/bin/printf', '']]

reference = {argv[0]: measure(argv) for argv in REFERENCES if Path(argv[0]).exists()}
FLOOR = min(reference.values(), key=lambda timing: timing.min)
SPAWN_FLOOR = FLOOR.min

for name, timing in sorted(reference.items(), key=lambda item: item[1].min):
    print(f'{name:<18} {timing.min:>6.2f} ms   (median {timing.median:>5.2f} ms)')
spread = max(t.min for t in reference.values()) - SPAWN_FLOOR
print(f'\nspawn floor        {SPAWN_FLOOR:>6.2f} ms, the cheapest of them')
print(f'they disagree by   {spread:>6.2f} ms, which is why none of them is subtracted')

/usr/bin/true        1.17 ms   (median  1.30 ms)
/bin/echo            1.22 ms   (median  1.44 ms)
/usr/bin/printf      1.28 ms   (median  1.45 ms)

spawn floor          1.17 ms, the cheapest of them
they disagree by     0.11 ms, which is why none of them is subtracted


In [3]:
import json

scratch = Path('/tmp/markdown-prose-bench')
scratch.mkdir(exist_ok=True)

# One hard-wrapped paragraph is the unit of work this tool exists to undo.
PARAGRAPH = 'A paragraph that has been hard\nwrapped across three\nseparate lines.\n\n'
(scratch / 'tiny.md').write_text(PARAGRAPH)

tracked = subprocess.run(
    ['git', 'ls-files', '*.md'], cwd=REPO, capture_output=True, text=True, check=True
).stdout.split()
(scratch / 'tracked.txt').write_text(
    '\n'.join(str(REPO / name) for name in tracked) + '\n'
)

# A synthetic tree, so file count varies independently of this repository.
bulk = scratch / 'bulk'
bulk.mkdir(exist_ok=True)
for index in range(2000):
    (bulk / f'{index}.md').write_text(PARAGRAPH * 3)

# One large document, which amortizes startup away and leaves throughput.
large = scratch / 'large.md'
large.write_text(PARAGRAPH * 60000)

# Passed explicitly because this notebook runs from `docs/`, and the tool reads
# `.unwrapignore` relative to the working directory. Without it the benchmark
# sweeps `corpus/`, which is fixture bytes rather than prose: 200-odd answer
# keys averaging a couple of dozen bytes, plus a deliberately non-UTF-8 case
# that the tool correctly refuses to read. That is not this repository's
# Markdown, and a mean taken over it is not this repository's file size.
IGNORE = ['--ignore-file', str(REPO / '.unwrapignore')]

# In scope means what the tool reports on, asked of the tool rather than
# reimplemented here -- the ignore rules are its own and it owns their meaning.
scope = json.loads(
    subprocess.run(
        [
            str(PYTHON_CLI),
            '--files-from',
            str(scratch / 'tracked.txt'),
            *IGNORE,
            '--json',
        ],
        capture_output=True,
        text=True,
        check=True,
    ).stdout
)
in_scope = [Path(entry['path']) for entry in scope['files']]
sizes = sorted(path.stat().st_size for path in in_scope)
REPO_MEAN_BYTES = sum(sizes) / len(sizes)

print(
    f'{len(tracked)} tracked Markdown files, {len(in_scope)} in scope and '
    f'{len(tracked) - len(in_scope)} excluded as corpus fixtures'
)
print(f'  in scope: {sum(sizes) / 1024:.0f} KB, mean {REPO_MEAN_BYTES:,.0f} bytes/file')
print(f'synthetic bulk file: {len(PARAGRAPH) * 3} bytes')
print(f'large document: {large.stat().st_size / 1e6:.1f} MB')

218 tracked Markdown files, 5 in scope and 213 excluded as corpus fixtures
  in scope: 170 KB, mean 34,860 bytes/file
synthetic bulk file: 207 bytes
large document: 4.1 MB


In [4]:
from IPython.display import Markdown, display


def file_list(count):
    listing = scratch / f'bulk-{count}.txt'
    listing.write_text(
        '\n'.join(str(bulk / f'{index}.md') for index in range(count)) + '\n'
    )
    return ['--files-from', str(listing)]


# The 4 MB row runs about a second per invocation in Python, so it takes fewer
# repetitions than the rest. Stated here rather than left for a reader to infer
# from a runtime.
LARGE_REPS = 10

scenarios = [
    ('startup floor: 1 file', [str(scratch / 'tiny.md')], REPS),
    ('a typical commit: 5 files', file_list(5), REPS),
    (
        f'this repository: {len(in_scope)} prose files at '
        f'{REPO_MEAN_BYTES / 1024:.0f} KB each',
        ['--files-from', str(scratch / 'tracked.txt'), *IGNORE],
        REPS,
    ),
    ('a large repository: 2000 files', file_list(2000), REPS),
    (f'one {large.stat().st_size / 1e6:.0f} MB document', [str(large)], LARGE_REPS),
]

results = []
for label, args, reps in scenarios:
    results.append(
        {
            'scenario': label,
            'python': measure([str(PYTHON_CLI), *args, '--json'], reps),
            'rust': measure([str(RUST_CLI), *args, '--json'], reps),
        }
    )


def ratio_cell(row):
    """Python over Rust, marked as a bound when the Rust arm sits on the floor."""
    value = row['python'].min / row['rust'].min
    mark = BOUND if row['rust'].min < SPAWN_FLOOR * FLOOR_FACTOR else ''
    return f'{mark}{value:.1f}x'


table = [
    '| scenario | Python min / median | Rust min / median | ratio |',
    '| -- | --: | --: | --: |',
]
for row in results:
    python, rust = row['python'], row['rust']
    table.append(
        f'| {row["scenario"]} '
        f'| {python.min:.1f} / {python.median:.1f} ms '
        f'| {rust.min:.1f} / {rust.median:.1f} ms '
        f'| {ratio_cell(row)} |'
    )
table.append('')
table.append(
    f'{BOUND} marks a ratio whose Rust arm is within {FLOOR_FACTOR:.0f}x of the '
    f'{SPAWN_FLOOR:.2f} ms spawn floor, which makes the figure a lower bound: most '
    'of what that arm reports is the cost of starting any process at all.'
)
display(Markdown(chr(10).join(table)))

# Three separate questions, because they have three separate answers and only
# the first one can void a recommendation.
#
#   AGREE     did the two implementations return the same bytes and the same
#             exit code? This is the premise the whole document rests on.
#   STEADY    did each command return the same thing on every repetition? It
#             may not: `git ls-files` names a live working tree, and another
#             session editing a Markdown file mid-run shows up here as a read
#             error in one repetition out of thirty.
#   CLEAN     did everything exit 0? A consistent non-zero exit is the tool
#             reporting a file it could not read, which is a fact about the
#             tree rather than about either implementation.
AGREE = all(
    row['python'].digests == row['rust'].digests
    and row['python'].codes == row['rust'].codes
    for row in results
)
unsteady = [
    (row['scenario'], impl)
    for row in results
    for impl in ('python', 'rust')
    if len(row[impl].digests) > 1 or len(row[impl].codes) > 1
]
CLEAN = all(
    row[impl].codes == frozenset({0}) for row in results for impl in ('python', 'rust')
)
noisy = [
    (row['scenario'], impl, row[impl].spread)
    for row in results
    for impl in ('python', 'rust')
    if row[impl].spread > SPREAD_LIMIT
]

print()
if AGREE:
    print('Both implementations returned identical bytes and identical exit codes on')
    print('every scenario above, so what separates them here is cost and nothing else.')
else:
    print('!! The implementations disagreed. Every recommendation below is void.')
    for row in results:
        if row['python'].digests != row['rust'].digests:
            print(f'     differing output: {row["scenario"]}')
        if row['python'].codes != row['rust'].codes:
            print(f'     differing exit:   {row["scenario"]}')

if unsteady:
    print('\n!! A command did not return the same thing on every repetition, so the')
    print('   tree changed underneath the benchmark. Timings stand; re-take these:')
    for label, impl in unsteady:
        print(f'     {impl:<7} {label}')
elif not CLEAN:
    print('\n   Exited non-zero on every repetition, identically in both')
    print('   implementations: the tool reported a file it could not read. That is a')
    print('   fact about this tree rather than a disagreement between the two.')
    for row in results:
        codes = row['python'].codes | row['rust'].codes
        if codes != frozenset({0}):
            print(f'     {row["scenario"]:<32} exit {sorted(codes)}')

if noisy:
    print(f'\n!! Median more than {SPREAD_LIMIT:.0f}% above minimum -- the machine was')
    print('   busy and these numbers should be re-taken:')
    for label, impl, percent in noisy:
        print(f'     {impl:<7} {label:<32} +{percent:.0f}%')
else:
    print(f'Every median is within {SPREAD_LIMIT:.0f}% of its minimum.')

| scenario | Python min / median | Rust min / median | ratio |
| -- | --: | --: | --: |
| startup floor: 1 file | 26.2 / 27.0 ms | 1.4 / 1.5 ms | ≥18.6x |
| a typical commit: 5 files | 26.8 / 27.6 ms | 1.5 / 1.6 ms | ≥18.4x |
| this repository: 5 prose files at 34 KB each | 61.7 / 63.0 ms | 3.5 / 3.6 ms | 17.8x |
| a large repository: 2000 files | 201.1 / 209.7 ms | 33.3 / 34.7 ms | 6.0x |
| one 4 MB document | 1063.2 / 1071.7 ms | 69.7 / 71.1 ms | 15.2x |

≥ marks a ratio whose Rust arm is within 2x of the 1.17 ms spawn floor, which makes the figure a lower bound: most of what that arm reports is the cost of starting any process at all.


Both implementations returned identical bytes and identical exit codes on
every scenario above, so what separates them here is cost and nothing else.
Every median is within 15% of its minimum.


## Where the time actually goes

The table above mixes two effects, and separating them is what makes a
per-channel recommendation possible at all. The next cell measures the floor
directly: the cheapest process this machine can spawn, a bare interpreter doing
nothing, that interpreter after importing this tool's module, and then each
command-line entry point on a single file.

The gap between the interpreter rows is what importing the tool costs. The gap
between the last two is what a hook invocation saves. The reference process at
the top is what neither implementation can go below.

In [5]:
import matplotlib.pyplot as plt

floor_rows = {
    'cheapest process': FLOOR,
    'bare interpreter': measure([sys.executable, '-c', 'pass']),
    'interpreter + this module': measure(
        [sys.executable, '-c', 'import markdown_prose_hooks.unwrap']
    ),
    'Python CLI, one file': measure(
        [str(PYTHON_CLI), str(scratch / 'tiny.md'), '--json']
    ),
    'Rust CLI, one file': measure([str(RUST_CLI), str(scratch / 'tiny.md'), '--json']),
}
for label, timing in floor_rows.items():
    print(f'{label:<28} {timing.min:>6.2f} ms   (median {timing.median:>6.2f} ms)')

python_startup = floor_rows['Python CLI, one file'].min
rust_startup = floor_rows['Rust CLI, one file'].min
saving = python_startup - rust_startup

print(f'\nmeasured ratio             {python_startup / rust_startup:>6.1f}x')
if rust_startup > SPAWN_FLOOR:
    corrected = (python_startup - SPAWN_FLOOR) / (rust_startup - SPAWN_FLOOR)
    print(f'with the floor off both    {corrected:>6.1f}x')
    print(
        f'   Shown, not used. The Rust arm is {rust_startup / SPAWN_FLOOR:.1f}x the'
        ' spawn floor, so that'
    )
    print('   correction divides by a small difference of two similar numbers and')
    print('   moves a long way on a floor that is itself uncertain. Read the')
    print('   measured ratio as a lower bound instead.')
else:
    print('   The Rust CLI came in at or under the cheapest reference process, so')
    print('   its own startup is below what this harness can resolve. The measured')
    print('   ratio is a lower bound and no correction is meaningful.')

print(f'\nper-invocation saving      {saving:>6.1f} ms')
print(f'at 100 invocations a day   {saving * 100 / 1000:>6.1f} s/day')
print('   This subtraction is safe where the ratio above is not: the harness')
print('   cost is the same in both arms, so it cancels in a difference.')

figure, axes = plt.subplots(figsize=(8, 3.2))
labels = list(floor_rows)
axes.barh(
    labels,
    [floor_rows[label].min for label in labels],
    color=['#999999', '#bbbbbb', '#bbbbbb', '#1f77b4', '#ff7f0e'],
)
axes.axvline(SPAWN_FLOOR, color='#444444', linestyle='--', linewidth=1)
axes.set_xscale('log')
axes.set_xlabel('wall clock (ms, log scale)')
axes.set_title('The dashed line is what starting any process costs')
axes.invert_yaxis()
for index, label in enumerate(labels):
    axes.text(
        floor_rows[label].min * 1.15,
        index,
        f'{floor_rows[label].min:.2f} ms',
        va='center',
        fontsize=9,
    )
axes.set_xlim(right=max(t.min for t in floor_rows.values()) * 3)
axes.grid(alpha=0.3, axis='x')
figure.tight_layout()
figure.savefig(REPO / 'docs/benchmarks-startup.svg')
plt.close(figure)
print('\nchart written to docs/benchmarks-startup.svg')

cheapest process               1.17 ms   (median   1.30 ms)
bare interpreter              12.68 ms   (median  13.07 ms)
interpreter + this module     25.14 ms   (median  25.65 ms)
Python CLI, one file          26.97 ms   (median  28.97 ms)
Rust CLI, one file             1.52 ms   (median   1.58 ms)

measured ratio               17.8x
with the floor off both      74.9x
   Shown, not used. The Rust arm is 1.3x the spawn floor, so that
   correction divides by a small difference of two similar numbers and
   moves a long way on a floor that is itself uncertain. Read the
   measured ratio as a lower bound instead.

per-invocation saving        25.5 ms
at 100 invocations a day      2.5 s/day
   This subtraction is safe where the ratio above is not: the harness
   cost is the same in both arms, so it cancels in a difference.

chart written to docs/benchmarks-startup.svg


![The dashed line is what starting any process costs](benchmarks-startup.svg)

## Throughput, with startup removed

One invocation over a growing number of files separates the constant cost from
the marginal one. On the left the intercept is startup and the slope is
throughput; on the right is the ratio between the two implementations at each
size, which is the same information asked the way a reader choosing between
them would ask it.

Both axes are logarithmic on the left panel because the counts span three
orders of magnitude, and a linear axis puts most of the measurements in the
first quarter of the plot where the intercept it is meant to show cannot be
read.

In [6]:
# Fewer repetitions than the headline table: these two sweeps exist to show the
# shape of a curve rather than to source a quoted figure, and the shape is
# already stable at this count.
SWEEP_REPS = 15

counts = [1, 10, 50, 100, 250, 500, 1000, 2000]
curve = {'Python': [], 'Rust': []}
for count in counts:
    args = file_list(count)
    curve['Python'].append(measure([str(PYTHON_CLI), *args, '--json'], SWEEP_REPS).min)
    curve['Rust'].append(measure([str(RUST_CLI), *args, '--json'], SWEEP_REPS).min)

figure, (left, right) = plt.subplots(1, 2, figsize=(11, 4.2))
for name, series in curve.items():
    left.plot(counts, series, marker='o', label=name)
left.axhline(SPAWN_FLOOR, color='#444444', linestyle='--', linewidth=1)
left.text(counts[0], SPAWN_FLOOR * 1.15, 'spawn floor', fontsize=8, color='#444444')
left.set_xscale('log')
left.set_yscale('log')
left.set_xlabel('files in one invocation')
left.set_ylabel('wall clock (ms)')
left.set_title('Startup is the intercept; throughput is the slope')
left.legend()
left.grid(alpha=0.3, which='both')

ratios = [p / r for p, r in zip(curve['Python'], curve['Rust'], strict=True)]
right.plot(counts, ratios, marker='o', color='#2ca02c')
right.set_xscale('log')
right.set_ylim(bottom=0)
right.set_xlabel('files in one invocation')
right.set_ylabel('Python / Rust')
right.set_title('One ratio, falling as per-file work dilutes startup')
right.grid(alpha=0.3, which='both')

# Written beside the notebook rather than embedded, and SVG rather than PNG.
# Embedding makes a base64 blob that gets spell-checked like prose, and
# codespell duly read one as a typo. PNG would be worse still: this
# repository's .gitattributes routes `*.png` through Git LFS, so a chart
# would commit as a pointer and show as a broken image to anyone cloning
# without git-lfs. SVG is text, so no rule touches it.
figure.tight_layout()
figure.savefig(REPO / 'docs/benchmarks.svg')
plt.close(figure)

# A two-point secant rather than a fit, which is only honest if the curve is
# straight between them. Checked rather than assumed: every consecutive pair is
# reported, and a fit would hide a bend that this shows.
count_marginal = {}
print(f'{"":<8} {"intercept":>10} {"marginal":>12}   consecutive-pair marginals')
for name, series in curve.items():
    pairs = [
        (series[i + 1] - series[i]) / (counts[i + 1] - counts[i]) * 1000
        for i in range(len(counts) - 1)
    ]
    count_marginal[name] = (series[-1] - series[0]) / (counts[-1] - counts[0]) * 1000
    print(
        f'{name:<8} {series[0]:>7.1f} ms {count_marginal[name]:>8.1f} us/file   '
        + ' '.join(f'{value:.0f}' for value in pairs)
    )
print('\nThe leftmost pairs are differences between measurements a few ms apart, so')
print('they carry most of the noise. They settle once per-file work clears the floor,')
print('and that settling is what makes the two-point marginal a fair summary of the')
print('slope rather than a fitted guess hiding a bend.')
print(
    f'\nratio at {counts[0]} file: {ratios[0]:.1f}x   '
    f'at {counts[-1]} files: {ratios[-1]:.1f}x'
)
print('chart written to docs/benchmarks.svg')

          intercept     marginal   consecutive-pair marginals
Python      27.0 ms     86.9 us/file   112 75 109 85 88 89 85
Rust         1.5 ms     16.2 us/file   73 2 16 14 16 16 17

The leftmost pairs are differences between measurements a few ms apart, so
they carry most of the noise. They settle once per-file work clears the floor,
and that settling is what makes the two-point marginal a fair summary of the
slope rather than a fitted guess hiding a bend.

ratio at 1 file: 17.5x   at 2000 files: 5.9x
chart written to docs/benchmarks.svg


![Startup is the intercept; throughput is the slope](benchmarks.svg)

## What the per-file multiplier depends on

The marginal cost of one more file is not a constant and is not a property of
the tool. Opening a file is a syscall, which costs about the same in either
language; transforming its contents is the work the two implementations do
differently. The multiplier a repository feels is set by the balance between
those two, which is to say by how many bytes its average Markdown file carries.

The next cell measures that directly. For each file size it times two
invocations, over a small and a large number of files of that size, and takes
the difference between them. Differencing removes startup exactly rather than
by estimate, which matters here: at the small-file end an un-differenced
measurement would be mostly startup, and startup is the one thing this cell is
trying to see past.

What comes out is one curve, running from the cost of barely more than opening
a file to the cost of the transform itself. Every per-file multiplier ever
quoted about this tool is a point on it, and which point applies to a given
repository is a question about that repository's Markdown rather than about the
tool. The marker shows where this one lands.

In [7]:
import numpy as np

# Two file counts per size, and the difference between them. Startup cancels
# exactly in that subtraction, so nothing here needs an estimate of it. That
# matters most at the small-file end, where an un-differenced measurement would
# be mostly startup -- the very thing this cell exists to look past.
SIZE_REPS = 8
LOW, HIGH = 100, 500
paragraph_counts = [1, 3, 10, 30, 100]


def listing(folder, count):
    path = scratch / f'{folder.name}-{count}.txt'
    path.write_text(
        '\n'.join(str(folder / f'{index}.md') for index in range(count)) + '\n'
    )
    return ['--files-from', str(path), '--json']


def marginal_us(cli, folder):
    """Microseconds one more file of this size costs, startup differenced away."""
    low = measure([str(cli), *listing(folder, LOW)], SIZE_REPS).min
    high = measure([str(cli), *listing(folder, HIGH)], SIZE_REPS).min
    return (high - low) / (HIGH - LOW) * 1000


sweep = {'bytes': [], 'Python': [], 'Rust': []}
for paragraphs in paragraph_counts:
    folder = scratch / f'size-{paragraphs}'
    folder.mkdir(exist_ok=True)
    for index in range(HIGH):
        (folder / f'{index}.md').write_text(PARAGRAPH * paragraphs)
    sweep['bytes'].append(len(PARAGRAPH) * paragraphs)
    sweep['Python'].append(marginal_us(PYTHON_CLI, folder))
    sweep['Rust'].append(marginal_us(RUST_CLI, folder))

size_ratios = [p / r for p, r in zip(sweep['Python'], sweep['Rust'], strict=True)]

# Interpolated in log space, because the samples are spaced logarithmically.
REPO_RATIO = float(
    np.interp(np.log10(REPO_MEAN_BYTES), np.log10(sweep['bytes']), size_ratios)
)
CLAMPED = not sweep['bytes'][0] <= REPO_MEAN_BYTES <= sweep['bytes'][-1]

# Pinned to the sampled range so an out-of-range mean annotates the plateau it
# extrapolates from rather than stretching the axis into empty space.
marker_x = min(max(REPO_MEAN_BYTES, sweep['bytes'][0]), sweep['bytes'][-1])
RATIO_TEXT = f'{REPO_RATIO:.1f}x or more' if CLAMPED else f'{REPO_RATIO:.1f}x'

figure, axes = plt.subplots(figsize=(8, 4.5))
axes.plot(sweep['bytes'], size_ratios, marker='o', color='#2ca02c')
axes.axvline(marker_x, color='#d62728', linestyle='--', linewidth=1)
axes.annotate(
    f'this repository\n{REPO_MEAN_BYTES:,.0f} B/file mean \u2192 {RATIO_TEXT}',
    xy=(marker_x, REPO_RATIO),
    # A clamped marker sits on the right edge, where a label offset outward
    # would be drawn off the canvas. Turned inward there instead.
    xytext=(-16, -78) if CLAMPED else (16, -34),
    textcoords='offset points',
    horizontalalignment='right' if CLAMPED else 'left',
    fontsize=9,
    color='#d62728',
    arrowprops={'arrowstyle': '->', 'color': '#d62728'},
)
axes.set_xscale('log')
axes.set_ylim(bottom=0)
axes.set_xlabel(
    f'bytes per file (log scale; each point differences {HIGH} files against {LOW})'
)
axes.set_ylabel('marginal cost, Python / Rust')
axes.set_title('The per-file multiplier is a function of how much prose a file holds')
axes.grid(alpha=0.3, which='both')
figure.tight_layout()
figure.savefig(REPO / 'docs/benchmarks-bytes.svg')
plt.close(figure)

table = [
    '| bytes/file | Python | Rust | ratio |',
    '| --: | --: | --: | --: |',
]
for size, python_us, rust_us, value in zip(
    sweep['bytes'], sweep['Python'], sweep['Rust'], size_ratios, strict=True
):
    table.append(
        f'| {size} | {python_us:.1f} us/file | {rust_us:.1f} us/file | {value:.1f}x |'
    )
table.append('')
table.append(
    f'Spanning {min(size_ratios):.1f}x to {max(size_ratios):.1f}x across the '
    f'{sweep["bytes"][-1] // sweep["bytes"][0]}x range of file sizes sampled. This '
    f'repository averages {REPO_MEAN_BYTES:,.0f} bytes a file, which lands at '
    f'**{RATIO_TEXT}**'
    + (
        ' -- past the largest size sampled, where the curve has already flattened,'
        ' so the true figure is at the plateau rather than beyond it.'
        if CLAMPED
        else '.'
    )
)
display(Markdown(chr(10).join(table)))

# The file-count sweep measured this same quantity at one point, since its
# files are 207 bytes each. Two independent routes to one number is a check
# worth printing rather than an agreement worth assuming.
common = sweep['bytes'].index(len(PARAGRAPH) * 3)
print(
    f'\nCross-check at {sweep["bytes"][common]} bytes/file, where the count sweep '
    'measured the same thing:'
)
for name in ('Python', 'Rust'):
    here, there = sweep[name][common], count_marginal[name]
    print(
        f'  {name:<7} {here:>6.1f} us/file here, {there:>6.1f} us/file there '
        f'({abs(here - there) / there * 100:>4.0f}% apart)'
    )
print('chart written to docs/benchmarks-bytes.svg')

| bytes/file | Python | Rust | ratio |
| --: | --: | --: | --: |
| 69 | 55.6 us/file | 13.5 us/file | 4.1x |
| 207 | 94.7 us/file | 15.3 us/file | 6.2x |
| 690 | 206.3 us/file | 23.4 us/file | 8.8x |
| 2070 | 555.0 us/file | 46.6 us/file | 11.9x |
| 6900 | 1776.1 us/file | 124.8 us/file | 14.2x |

Spanning 4.1x to 14.2x across the 100x range of file sizes sampled. This repository averages 34,860 bytes a file, which lands at **14.2x or more** -- past the largest size sampled, where the curve has already flattened, so the true figure is at the plateau rather than beyond it.


Cross-check at 207 bytes/file, where the count sweep measured the same thing:
  Python    94.7 us/file here,   86.9 us/file there (   9% apart)
  Rust      15.3 us/file here,   16.2 us/file there (   5% apart)
chart written to docs/benchmarks-bytes.svg


![The per-file multiplier is a function of how much prose a file holds](benchmarks-bytes.svg)

## How to choose

The cell below assembles the recommendation from the measurements above and
from the current state of the repository. It reads `action.yml` to see which
implementation that channel actually runs today, and it checks whether a
release exists, because two of the rows describe things that a first tag
unblocks and that no amount of work in this notebook can.

In [8]:
import textwrap

action = (REPO / 'action.yml').read_text()
ACTION_USES_BINARY = 'setup-python' not in action
RELEASED = bool(TAGS)

startup_ratio = python_startup / rust_startup
bound = BOUND if rust_startup < SPAWN_FLOOR * FLOOR_FACTOR else ''
transform_ratio = max(size_ratios)
binary_kb = RUST_CLI.stat().st_size / 1024

# The repository marks a configuration file it cannot yet give its intended
# shape with this string, and says in the comment beneath it what is blocked
# and on what. Scanned rather than restated: a list of caveats maintained in
# this notebook would be a second copy to keep in step, and the copy nobody
# edits is the one that goes stale.
MARKER = 'DEVIATION, blocked on'


def deviation(path):
    """The comment block introduced by MARKER in `path`, as one paragraph."""
    lines = path.read_text().splitlines()
    for index, line in enumerate(lines):
        if MARKER in line:
            block = []
            for follow in lines[index:]:
                stripped = follow.lstrip()
                if not stripped.startswith('#'):
                    break
                block.append(stripped.lstrip('#').strip())
            return ' '.join(part for part in block if part)
    return ''


pending = []
for name in ('action.yml', '.pre-commit-hooks.yaml', 'pyproject.toml', 'Cargo.toml'):
    note = deviation(REPO / name)
    if note:
        # The first sentence is the marker itself, which the heading above
        # already says; the two after it are what the deviation actually is.
        sentences = note.split('. ')[1:3]
        pending.append(f'**`{name}`** -- ' + '. '.join(sentences).rstrip('.') + '.')

if ACTION_USES_BINARY:
    action_use = '`-rs`'
    action_why = (
        f'The runner is provisioned fresh every job, so a prebuilt {binary_kb:.0f} KB '
        'binary installs faster than either toolchain provisions and runs faster '
        'afterwards. The action downloads one.'
    )
else:
    action_use = '`-py`, which is what the action runs today'
    action_why = (
        'The action provisions Python and pip-installs the package, so this channel '
        'offers no choice yet. A prebuilt binary would be faster to install *and* to '
        'run, but '
        + (
            f'the action has not been switched over since {TAGS[-1]} landed.'
            if RELEASED
            else 'the action has nothing to download from until a tag exists.'
        )
    )

cli_install = (
    'Install with `pipx install markdown-prose-hooks`.'
    if RELEASED
    else 'Installed from a checkout for now; nothing reaches PyPI until the first tag.'
)
if not RELEASED:
    pending.append(
        '**registries** -- neither PyPI nor crates.io carries this package yet, so '
        'the command-line rows above describe installing from a checkout.'
    )

rows = [
    (
        '`pre-commit`, ordinary repository',
        '`-py`',
        f'The whole cost is startup, at {bound}{startup_ratio:.0f}x and '
        f'{saving:.0f} ms an invocation. `pre-commit` is itself a Python '
        'application, so the interpreter is already there; the Rust hook builds '
        'from source and needs cargo.',
    ),
    (
        '`pre-commit`, monorepo or slow `--all-files`',
        '`-rs`',
        f'A sweep pays the marginal per-file cost once per file, and that is a '
        f'function of file size rather than a constant: {min(size_ratios):.1f}x when '
        f'files are near-empty, {transform_ratio:.0f}x once they hold real prose. '
        f'This repository averages {REPO_MEAN_BYTES:,.0f} bytes a file, which puts it '
        f'at {RATIO_TEXT}. A from-source build is paid once.',
    ),
    ('GitHub Action', action_use, action_why),
    (
        'A script or CI step with no Python',
        '`-rs`',
        'A static binary with no runtime to provision.',
    ),
    ('Command line, Python already present', '`-py`', cli_install),
]

document = ['| how you run it | use | why |', '| -- | -- | -- |']
document += [f'| {channel} | {use} | {why} |' for channel, use, why in rows]

document += [
    '',
    '### Is the difference worth caring about?',
    '',
    textwrap.fill(
        f'Per invocation the saving is {saving:.0f} ms, or about '
        f'{saving * 100 / 1000:.1f} seconds across a hundred invocations a day. That '
        'compounds across a team, and agent-driven work commits far more often than '
        'human-driven work does, but on its own it is not a reason to adopt a '
        'toolchain you do not otherwise want.',
        width=86,
    ),
    '',
    textwrap.fill(
        'Where it stops being marginal is a sweep over thousands of files, which pays '
        f'the marginal multiplier -- {REPO_RATIO:.1f}x at the average file '
        f'size here -- once per file, rather than the {saving:.0f} ms once.',
        width=86,
    ),
]

if pending:
    document += [
        '',
        '### Blocked on the first release',
        '',
        textwrap.fill(
            'Read as: what this notebook measured is what ships today, and these are '
            'the places where what ships is not yet what is designed. Each is quoted '
            'from the file that carries it.',
            width=86,
        ),
        '',
    ]
    document += [f'- {note}' for note in pending]

if not AGREE:
    document += [
        '',
        '**The implementations disagreed further up. Disregard all of the above.**',
    ]

display(Markdown(chr(10).join(document)))

| how you run it | use | why |
| -- | -- | -- |
| `pre-commit`, ordinary repository | `-py` | The whole cost is startup, at ≥18x and 25 ms an invocation. `pre-commit` is itself a Python application, so the interpreter is already there; the Rust hook builds from source and needs cargo. |
| `pre-commit`, monorepo or slow `--all-files` | `-rs` | A sweep pays the marginal per-file cost once per file, and that is a function of file size rather than a constant: 4.1x when files are near-empty, 14x once they hold real prose. This repository averages 34,860 bytes a file, which puts it at 14.2x or more. A from-source build is paid once. |
| GitHub Action | `-py`, which is what the action runs today | The action provisions Python and pip-installs the package, so this channel offers no choice yet. A prebuilt binary would be faster to install *and* to run, but the action has nothing to download from until a tag exists. |
| A script or CI step with no Python | `-rs` | A static binary with no runtime to provision. |
| Command line, Python already present | `-py` | Installed from a checkout for now; nothing reaches PyPI until the first tag. |

### Is the difference worth caring about?

Per invocation the saving is 25 ms, or about 2.5 seconds across a hundred invocations
a day. That compounds across a team, and agent-driven work commits far more often than
human-driven work does, but on its own it is not a reason to adopt a toolchain you do
not otherwise want.

Where it stops being marginal is a sweep over thousands of files, which pays the
marginal multiplier -- 14.2x at the average file size here -- once per file, rather
than the 25 ms once.

### Blocked on the first release

Read as: what this notebook measured is what ships today, and these are the places
where what ships is not yet what is designed. Each is quoted from the file that
carries it.

- **`action.yml`** -- This provisions Python and runs the `-py` implementation, which is not the intended shape. The design calls for downloading a prebuilt binary from the release for this tag -- roughly a megabyte, no toolchain, faster to install as well as to run -- with an `implementation` input taking `auto`, `rust` or `python`, and `pip install` kept as the fallback for a runner with no published binary.
- **`.pre-commit-hooks.yaml`** -- This file is meant to go away. At `v0.1.0` each pair moves to a mirror repository serving only its own implementation -- `markdown-prose-hooks-py` and `markdown-prose-hooks-rs` -- so a consumer stops cloning roughly 1.2 MB carrying both plus 355 corpus fixtures to get one of them.
- **registries** -- neither PyPI nor crates.io carries this package yet, so the command-line rows above describe installing from a checkout.